# Quick LOCC with TF-IDF

Minimal laptop-friendly experiment for **Latent Ordinal Corruption Channels (LOCC)** using `data/train_lang.csv`.

This notebook avoids transformers on purpose: it uses TF-IDF + linear classifiers to approximate the surface sentiment model and the main `P(z | x)` model, then builds LOCC posterior soft labels and trains a small soft-label linear classifier with SGD.

## Setup

Set `SAMPLE_N = None` to use all 252k rows. The default sample should run quickly on a notebook while still being large enough to make the noise heuristics interesting.

In [5]:
from pathlib import Path

import numpy as np
import pandas as pd
from scipy.sparse import vstack
from scipy.special import softmax
from sklearn.base import clone
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import SGDClassifier
from sklearn.metrics import accuracy_score, mean_absolute_error, cohen_kappa_score, classification_report
from sklearn.model_selection import train_test_split

RANDOM_STATE = 42
DATA_PATH = Path("../data/train_lang.csv") if Path("../data/train_lang.csv").exists() else Path("data/train_lang.csv")
SAMPLE_N = None  # set to None for the full train set
TEST_SIZE = 0.10
N_CLASSES = 5
LABELS = np.arange(N_CLASSES)

In [6]:
df = pd.read_csv(DATA_PATH)
df["sentence"] = df["sentence"].fillna("")
df["lang"] = df["lang"].fillna("unk")

if SAMPLE_N is not None and SAMPLE_N < len(df):
    df, _ = train_test_split(
        df,
        train_size=SAMPLE_N,
        random_state=RANDOM_STATE,
        stratify=df["label"],
    )
    df = df.reset_index(drop=True)

df["text"] = "[lang=" + df["lang"].astype(str) + "] " + df["sentence"].astype(str)
df.head()

,id,sentence,label,lang,text
0,0,Moderner Weihnachtsbaum in weiß\n\nDieses Jahr...,4,deu_Latn,[lang=deu_Latn] Moderner Weihnachtsbaum in wei...
1,1,Passt wie angegossen\n\nTasche kam schnell und...,4,deu_Latn,[lang=deu_Latn] Passt wie angegossen\n\nTasche...
2,2,Schlechte Qualität\n\nIch habe sehr lange auf ...,1,deu_Latn,[lang=deu_Latn] Schlechte Qualität\n\nIch habe...
3,3,Bestellung nie angekommen\n\n-5 Sterne..... Am...,0,deu_Latn,[lang=deu_Latn] Bestellung nie angekommen\n\n-...
4,4,Für mich gar nicht gut\n\nMacht das Makeup gar...,1,deu_Latn,[lang=deu_Latn] Für mich gar nicht gut\n\nMach...


In [7]:
train_df, valid_df = train_test_split(
    df,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE,
    stratify=df["label"],
)

X_text_train = train_df["text"].to_numpy()
X_text_valid = valid_df["text"].to_numpy()
y_train = train_df["label"].to_numpy(dtype=int)
y_valid = valid_df["label"].to_numpy(dtype=int)

print(train_df.shape, valid_df.shape)
print(train_df["label"].value_counts().sort_index().to_dict())
print(train_df["lang"].value_counts().to_dict())

(226800, 5) (25200, 5)
{0: 45360, 1: 45360, 2: 45360, 3: 45360, 4: 45360}
{'deu_Latn': 113600, 'eng_Latn': 113200}


## Baseline: TF-IDF + SGD logistic regression

This is intentionally simple. Character n-grams usually work well for mixed English/German informal text, misspellings, and compound words.

In [8]:
vectorizer = TfidfVectorizer(
    analyzer="char_wb",
    ngram_range=(3, 5),
    min_df=3,
    max_features=200_000,
    sublinear_tf=True,
    dtype=np.float32,
)

X_train = vectorizer.fit_transform(X_text_train)
X_valid = vectorizer.transform(X_text_valid)
X_train.shape, X_valid.shape

((226800, 200000), (25200, 200000))

In [9]:
def evaluate(name, y_true, y_pred):
    print(f"{name}")
    print(f"  MAE:      {mean_absolute_error(y_true, y_pred):.4f}")
    print(f"  Accuracy: {accuracy_score(y_true, y_pred):.4f}")
    print(f"  QWK:      {cohen_kappa_score(y_true, y_pred, weights='quadratic'):.4f}")


base_clf = SGDClassifier(
    loss="log_loss",
    penalty="l2",
    alpha=1e-5,
    max_iter=8,
    tol=1e-3,
    n_jobs=-1,
    random_state=RANDOM_STATE,
)
base_clf.fit(X_train, y_train)

base_valid_proba = base_clf.predict_proba(X_valid)
base_valid_pred = base_valid_proba.argmax(axis=1)
evaluate("Baseline TF-IDF + SGD", y_valid, base_valid_pred)

Baseline TF-IDF + SGD
  MAE:      0.5411
  Accuracy: 0.5803
  QWK:      0.7970


/home/micha/anaconda3/envs/cil_fresh/lib/python3.11/site-packages/sklearn/linear_model/_stochastic_gradient.py:733: ConvergenceWarning: Maximum number of iteration reached before convergence. Consider increasing max_iter to improve the fit.
  warnings.warn(


## Surface sentiment model

For a cheap `v = h(x)`, train a deliberately smaller TF-IDF model. Its mistakes are useful: LOCC treats disagreement between this surface model, the observed label, and the main model as evidence for possible corruption modes.

In [10]:
surface_vectorizer = TfidfVectorizer(
    analyzer="word",
    ngram_range=(1, 2),
    min_df=3,
    max_features=80_000,
    sublinear_tf=True,
    dtype=np.float32,
)
X_surface_train = surface_vectorizer.fit_transform(X_text_train)
X_surface_valid = surface_vectorizer.transform(X_text_valid)

surface_clf = clone(base_clf).set_params(alpha=3e-5, max_iter=6, random_state=RANDOM_STATE + 1)
surface_clf.fit(X_surface_train, y_train)

v_train = surface_clf.predict(X_surface_train).astype(int)
v_valid = surface_clf.predict(X_surface_valid).astype(int)
evaluate("Surface word TF-IDF model", y_valid, v_valid)

Surface word TF-IDF model
  MAE:      0.5207
  Accuracy: 0.5929
  QWK:      0.8053


/home/micha/anaconda3/envs/cil_fresh/lib/python3.11/site-packages/sklearn/linear_model/_stochastic_gradient.py:733: ConvergenceWarning: Maximum number of iteration reached before convergence. Consider increasing max_iter to improve the fit.
  warnings.warn(


## LOCC channels and heuristic gate

The posterior is:

`q(z) proportional to P_theta(z | x) * sum_m pi(m | s_i) K_m(y, v | z)`

Here `pi(m | s_i)` is hand-designed for speed. This is the part to replace later with a learned gating model or BERT-derived loss/gradient features.

In [11]:
def ordinal_kernel(observed, z, lam=1.35):
    numer = np.exp(-lam * np.abs(observed - z))
    denom = np.exp(-lam * np.abs(LABELS - z)).sum()
    return numer / denom


def sarcasm_surface_kernel(v, z, alpha=1.20):
    opposite = (N_CLASSES - 1) - z
    numer = np.exp(-alpha * np.abs(v - opposite))
    denom = np.exp(-alpha * np.abs(LABELS - opposite)).sum()
    return numer / denom


def corruption_gate(y, v, pred, loss, max_prob):
    # Columns: clean, mislabel, default0, sarcasm
    pi = np.tile(np.array([0.72, 0.16, 0.06, 0.06], dtype=np.float32), (len(y), 1))
    d_yv = np.abs(y - v)
    d_yp = np.abs(y - pred)
    high_loss = loss > np.quantile(loss, 0.75)
    low_conf = max_prob < np.quantile(max_prob, 0.35)

    default0 = (y == 0) & (v >= 3) & (pred >= 3)
    sarcasm = (d_yv >= 3) & (pred == y)
    mislabel = high_loss & (d_yv >= 2) & (d_yp >= 2)
    clean = (d_yv <= 1) & (pred == y) & ~low_conf

    pi[default0, 2] += 1.50
    pi[sarcasm, 3] += 0.90
    pi[mislabel, 1] += 1.00
    pi[clean, 0] += 0.50

    return pi / pi.sum(axis=1, keepdims=True)


def locc_soft_labels(main_proba, y, v, eps=1e-12):
    pred = main_proba.argmax(axis=1)
    max_prob = main_proba.max(axis=1)
    loss = -np.log(main_proba[np.arange(len(y)), y] + eps)
    pi = corruption_gate(y, v, pred, loss, max_prob)

    q = np.zeros((len(y), N_CLASSES), dtype=np.float32)
    channels = np.zeros((len(y), N_CLASSES), dtype=np.float32)

    for z in LABELS:
        k_clean = (y == z).astype(np.float32)
        k_mis = ordinal_kernel(y, z).astype(np.float32)
        k_default0 = (y == 0).astype(np.float32)
        k_sarc = ((y == z).astype(np.float32) * sarcasm_surface_kernel(v, z)).astype(np.float32)

        channels[:, z] = (
            pi[:, 0] * k_clean
            + pi[:, 1] * k_mis
            + pi[:, 2] * k_default0
            + pi[:, 3] * k_sarc
        )
        q[:, z] = main_proba[:, z] * channels[:, z]

    q = q + eps
    q = q / q.sum(axis=1, keepdims=True)
    diagnostics = pd.DataFrame(
        {
            "y": y,
            "v_surface": v,
            "pred_main": pred,
            "loss": loss,
            "pmax": max_prob,
            "pi_clean": pi[:, 0],
            "pi_mislabel": pi[:, 1],
            "pi_default0": pi[:, 2],
            "pi_sarcasm": pi[:, 3],
            "soft_argmax": q.argmax(axis=1),
            "soft_expected": q @ LABELS,
        },
        index=train_df.index,
    )
    return q, diagnostics


base_train_proba = base_clf.predict_proba(X_train)
q_train, locc_diag = locc_soft_labels(base_train_proba, y_train, v_train)
locc_diag.head()

,y,v_surface,pred_main,loss,pmax,pi_clean,pi_mislabel,pi_default0,pi_sarcasm,soft_argmax,soft_expected
235904,3,3,3,0.461181,0.630539,0.813333,0.106667,0.04,0.04,3,3.003316
45378,2,4,4,1.329669,0.311824,0.360000,0.580000,0.03,0.03,2,2.123312
250228,2,2,2,0.846673,0.428839,0.813333,0.106667,0.04,0.04,2,2.000306
79099,1,1,1,0.493923,0.610228,0.813333,0.106667,0.04,0.04,1,0.997760
156068,0,0,0,0.133276,0.875224,0.813333,0.106667,0.04,0.04,0,0.015737


In [12]:
print("Average corruption probabilities:")
display(locc_diag[["pi_clean", "pi_mislabel", "pi_default0", "pi_sarcasm"]].mean().to_frame("mean"))

print("Hard labels changed by LOCC soft-label argmax:")
print((locc_diag["y"] != locc_diag["soft_argmax"]).mean())

inspect_cols = ["sentence", "label", "lang"]
interesting = locc_diag.assign(abs_shift=np.abs(locc_diag["soft_expected"] - locc_diag["y"])).sort_values("abs_shift", ascending=False).head(12)
display(train_df.loc[interesting.index, inspect_cols].join(interesting))

Average corruption probabilities:


,mean
pi_clean,0.743072
pi_mislabel,0.156392
pi_default0,0.049864
pi_sarcasm,0.050672


Hard labels changed by LOCC soft-label argmax:
0.004448853615520282


,sentence,label,lang,y,v_surface,pred_main,loss,pmax,pi_clean,pi_mislabel,pi_default0,pi_sarcasm,soft_argmax,soft_expected,abs_shift
242023,"Five Stars\n\nperfect size, the best protein, ...",0,eng_Latn,0,4,4,5.702480,0.915844,0.205714,0.331429,0.445714,0.017143,4,3.829112,3.829112
212097,Super\n\nAlles super und lecker,0,deu_Latn,0,4,4,7.094978,0.719106,0.205714,0.331429,0.445714,0.017143,4,3.705627,3.705627
187906,"Top Qualität\n\nSuper toll. Fühlt sich gut an,...",0,deu_Latn,0,4,4,6.154177,0.751388,0.205714,0.331429,0.445714,0.017143,4,3.665234,3.665234
18410,sehr gute Qualität\n\ndas Produkt entsprach me...,0,deu_Latn,0,4,4,5.305543,0.777118,0.205714,0.331429,0.445714,0.017143,4,3.663881,3.663881
78023,Gern wieder!\n\nAlles super wie immer Danke!,0,deu_Latn,0,4,4,3.926336,0.785798,0.205714,0.331429,0.445714,0.017143,4,3.643981,3.643981
103976,perfekter Empfang\n\nsuper Empfang mit dem Ding,0,deu_Latn,0,4,4,4.039159,0.818783,0.205714,0.331429,0.445714,0.017143,4,3.610521,3.610521
68042,"Perfekt\n\nSehr, sehr gute Qualität. Dunkelt a...",0,deu_Latn,0,4,4,4.353883,0.757243,0.205714,0.331429,0.445714,0.017143,4,3.602794,3.602794
103374,Preis Leistungsverhältnis ist in Ordnung\n\nIc...,0,deu_Latn,0,4,4,5.298783,0.649437,0.205714,0.331429,0.445714,0.017143,4,3.580987,3.580987
68587,Works great\n\nThis really works makes you Eat...,0,eng_Latn,0,4,4,4.875306,0.676124,0.205714,0.331429,0.445714,0.017143,4,3.551587,3.551587
158381,PC Gehäuse\n\nAlles super immer wieder,0,deu_Latn,0,4,4,3.776447,0.761744,0.205714,0.331429,0.445714,0.017143,4,3.548843,3.548843


## Retrain fast model on LOCC soft labels

`SGDClassifier.partial_fit` accepts sample weights, so we can train one binary-vs-rest classifier per class using the posterior probability `q(z)` as the weight. This is a compact approximation of soft cross-entropy for a linear model.

In [13]:
def fit_soft_ovr_sgd(X, q, epochs=4, alpha=1e-5, random_state=42):
    X_aug = vstack([X, X], format="csr")
    clfs = []
    binary_classes = np.array([0, 1])
    for z in LABELS:
        clf = SGDClassifier(
            loss="log_loss",
            penalty="l2",
            alpha=alpha,
            max_iter=1,
            tol=None,
            random_state=random_state + int(z),
        )
        y_aug = np.r_[np.ones(X.shape[0], dtype=int), np.zeros(X.shape[0], dtype=int)]
        w_aug = np.r_[q[:, z], 1.0 - q[:, z]]
        w_aug = np.maximum(w_aug, 1e-4).astype(np.float64)
        clf.partial_fit(X_aug, y_aug, classes=binary_classes, sample_weight=w_aug)
        for _ in range(epochs - 1):
            clf.partial_fit(X_aug, y_aug, sample_weight=w_aug)
        clfs.append(clf)
    return clfs


def predict_soft_ovr_proba(clfs, X):
    scores = []
    for clf in clfs:
        if hasattr(clf, "decision_function"):
            scores.append(clf.decision_function(X))
        else:
            scores.append(clf.predict_proba(X)[:, 1])
    scores = np.vstack(scores).T
    return softmax(scores, axis=1)


locc_clfs = fit_soft_ovr_sgd(X_train, q_train, epochs=5, alpha=1e-5, random_state=RANDOM_STATE + 10)
locc_valid_proba = predict_soft_ovr_proba(locc_clfs, X_valid)
locc_valid_pred = locc_valid_proba.argmax(axis=1)

evaluate("LOCC soft-label TF-IDF + SGD", y_valid, locc_valid_pred)
print()
print(classification_report(y_valid, locc_valid_pred, digits=3))

LOCC soft-label TF-IDF + SGD
  MAE:      0.5481
  Accuracy: 0.5783
  QWK:      0.7898

              precision    recall  f1-score   support

           0      0.648     0.709     0.677      5040
           1      0.479     0.468     0.473      5040
           2      0.499     0.418     0.455      5040
           3      0.543     0.499     0.520      5040
           4      0.680     0.798     0.734      5040

    accuracy                          0.578     25200
   macro avg      0.570     0.578     0.572     25200
weighted avg      0.570     0.578     0.572     25200



## Optional: ordinal prediction by expected rating

For MAE, the expected rating can sometimes be better than argmax. This rounds `E[z]` to the nearest valid class.

In [14]:
base_expected_pred = np.rint(base_valid_proba @ LABELS).clip(0, 4).astype(int)
locc_expected_pred = np.rint(locc_valid_proba @ LABELS).clip(0, 4).astype(int)

evaluate("Baseline expected-rating rounded", y_valid, base_expected_pred)
evaluate("LOCC expected-rating rounded", y_valid, locc_expected_pred)

Baseline expected-rating rounded
  MAE:      0.5995
  Accuracy: 0.4685
  QWK:      0.7567
LOCC expected-rating rounded
  MAE:      0.5541
  Accuracy: 0.5207
  QWK:      0.7874


## Next experiments

- Replace the hand gate with a small multinomial logistic gate over `[abs(y-v), y==0, loss, max_prob, pred, lang]`.
- Use out-of-fold predictions for `v` and `P(z | x)` to reduce in-sample optimism.
- Swap the cheap `P(z | x)` with saved probabilities from LoRA mBERT/XLM-R.
- Add a real gradient-conflict feature from the transformer run and feed it into `corruption_gate`.